# Agentic Debugging QLoRA Patch Pilot v1 — Final Training Run

Authorized scope: **final training only** (one bounded, descriptive one-epoch QLoRA run on the
frozen 1,000-train / 150-validation corpus). Held-out generation and base-versus-tuned evaluation
remain **unauthorized**; no held-out task content is loaded anywhere in this notebook. Every attempt
uses a fresh isolated run directory under `final-training/runs/<run-id>/`; existing output is never
reused. This notebook is executed by the owner in Colab after FirstMate review; it does not rebuild
or top up the corpus.


In [ ]:
# Cell 1 — Install frozen user-space dependencies. Do not pin Colab's CUDA torch wheel.
%pip install -q \
  transformers==5.14.1 \
  datasets==5.0.0 \
  peft==0.20.0 \
  trl==1.8.0 \
  bitsandbytes==0.49.2 \
  accelerate==1.14.0 \
  huggingface_hub \
  safetensors


In [ ]:
# Cell 2 — Mount persistent external storage and identify the repository.
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
REPOSITORY_ROOT = Path('/content/agentic-debugging-internship')
DRIVE_ROOT = Path('/content/drive/MyDrive/agentic-debugging/qlora_patch_pilot_v1')
CORPUS_ROOT = DRIVE_ROOT / 'corpus'
RUNS_ROOT = DRIVE_ROOT / 'final-training' / 'runs'
MODEL_CACHE = DRIVE_ROOT / 'model-cache'
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_CACHE.mkdir(parents=True, exist_ok=True)
assert (REPOSITORY_ROOT / 'experiments/qlora_patch_pilot_v1/freeze_record.json').is_file()
assert (CORPUS_ROOT / 'train.jsonl').is_file() and (CORPUS_ROOT / 'validation.jsonl').is_file()
%cd {REPOSITORY_ROOT}


In [ ]:
# Cell 3 — Verify all frozen local identities before data or model work.
import json, subprocess, sys
FREEZE = REPOSITORY_ROOT / 'experiments/qlora_patch_pilot_v1/freeze_record.json'
result = subprocess.run([
    sys.executable, 'scripts/qlora_patch_pilot.py', 'verify-freeze',
    '--repository-root', str(REPOSITORY_ROOT), '--freeze-record', str(FREEZE),
], check=True, text=True, capture_output=True)
verification = json.loads(result.stdout)
freeze = json.loads(FREEZE.read_text(encoding='utf-8'))
assert verification['status'] == 'LOCKED' and verification['failed'] == []
assert freeze['repository_baseline']['base_commit'] == '66fb5d5'
assert freeze['repository_baseline']['relationship'] == 'required_ancestor'
assert freeze['scientific_gate']['final_training_authorized'] is False
assert freeze['scientific_gate']['held_out_generation_authorized'] is False
print(json.dumps(verification['runtime'], indent=2))


In [ ]:
# Cell 4 — Validate the final-training authorization against the exact real files (fail closed).
AUTHORIZATION = REPOSITORY_ROOT / 'experiments/qlora_patch_pilot_v1/final_training_authorization.json'
TRAIN_JSONL = CORPUS_ROOT / 'train.jsonl'
VALIDATION_JSONL = CORPUS_ROOT / 'validation.jsonl'
CORPUS_MANIFEST = CORPUS_ROOT / 'external_artifacts.json'
COMPLETED_AUDIT_CSV = DRIVE_ROOT / 'independent-audit' / 'firstmate_independent_audit_completed.csv'
COMPLETED_AUDIT_MANIFEST = DRIVE_ROOT / 'independent-audit' / 'firstmate_independent_audit_manifest.json'
for artifact in (AUTHORIZATION, TRAIN_JSONL, VALIDATION_JSONL, CORPUS_MANIFEST, COMPLETED_AUDIT_CSV, COMPLETED_AUDIT_MANIFEST):
    assert artifact.is_file(), f'required artifact missing: {artifact}'
auth_command = [
    sys.executable, 'scripts/qlora_patch_pilot.py', 'validate-final-training-auth',
    '--authorization', str(AUTHORIZATION),
    '--repository-root', str(REPOSITORY_ROOT),
    '--corpus-dir', str(CORPUS_ROOT),
    '--transformation-config', str(REPOSITORY_ROOT / 'experiments/qlora_patch_pilot_v1/transformation_config.json'),
    '--train-jsonl', str(TRAIN_JSONL),
    '--validation-jsonl', str(VALIDATION_JSONL),
    '--corpus-manifest', str(CORPUS_MANIFEST),
    '--completed-audit-csv', str(COMPLETED_AUDIT_CSV),
    '--completed-audit-manifest', str(COMPLETED_AUDIT_MANIFEST),
]
result = subprocess.run(auth_command, check=True, text=True, capture_output=True)
auth_result = json.loads(result.stdout)
assert auth_result['status'] == 'COMPLETE'
assert auth_result['authorization_scope'] == 'final_training_only'
assert auth_result['authorized'] is True
assert auth_result['held_out_generation_authorized'] is False
assert auth_result['base_versus_tuned_evaluation_authorized'] is False
assert auth_result['audit_mode'] == 'independent_ai'
assert auth_result['reviewer_identity'] == 'FirstMate / GPT-5.6 Thinking'
assert auth_result['reviewer_type'] == 'independent_ai_reviewer'
assert auth_result['row_counts'] == {'train': 1000, 'validation': 150}
assert auth_result['audit_counts']['total_rows'] == 75
assert auth_result['audit_counts']['accepted_packet_rows'] == 50
assert auth_result['audit_counts']['rejected_packet_rows'] == 25
assert auth_result['audit_counts']['accepted_packet_accept'] == 39
assert auth_result['audit_counts']['accepted_packet_reject'] == 11
assert auth_result['audit_counts']['rejected_packet_accept'] == 0
assert auth_result['audit_counts']['rejected_packet_reject'] == 25
print(result.stdout)


In [ ]:
# Cell 5 — Fresh isolated run identity and directory (never reuse existing output).
import datetime as _dt, hashlib as _hl
authorization_sha256 = _hl.sha256(AUTHORIZATION.read_bytes()).hexdigest()
run_id = _dt.datetime.now(_dt.timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + authorization_sha256[:8]
RUN_DIR = RUNS_ROOT / run_id
assert not RUN_DIR.exists(), f'run directory already exists; a fresh run id is required: {RUN_DIR}'
RUN_DIR.mkdir(parents=True)
run_context = {
    'run_id': run_id,
    'authorization_path': str(AUTHORIZATION),
    'authorization_sha256': authorization_sha256,
    'created_at_utc': _dt.datetime.now(_dt.timezone.utc).isoformat(),
    'status': 'INCOMPLETE',
}
(RUN_DIR / 'run_context.json').write_text(json.dumps(run_context, indent=2, sort_keys=True) + '\n')
(RUN_DIR / 'INCOMPLETE').write_text('INCOMPLETE run; see run_status.json\n', encoding='utf-8')
print(run_context)


In [ ]:
# Cell 6 — Verify the frozen corpus records; never rebuild or top up.
corpus_summary = json.loads((CORPUS_ROOT / 'corpus_summary.json').read_text(encoding='utf-8'))
dedup_report = json.loads((CORPUS_ROOT / 'dedup_report.json').read_text(encoding='utf-8'))
assert corpus_summary['corpus_tier'] == 'minimum'
assert corpus_summary['train_examples'] == 1000
assert corpus_summary['validation_examples'] == 150
assert dedup_report['repository_overlap'] == []
assert dedup_report['held_out_exact_matches_accepted'] == 0
assert dedup_report['held_out_near_matches_accepted'] == 0
print(corpus_summary)


In [ ]:
# Cell 7 — Record runtime identity and require a CUDA accelerator.
import importlib.metadata as md, platform, time
import torch
runtime_started = time.time()
runtime = {
    'python': platform.python_version(),
    'torch': torch.__version__,
    'cuda_available': torch.cuda.is_available(),
    'cuda_runtime': torch.version.cuda,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'gpu_total_memory_bytes': torch.cuda.get_device_properties(0).total_memory if torch.cuda.is_available() else None,
    'packages': {name: md.version(name) for name in ['transformers','datasets','peft','trl','bitsandbytes','accelerate','huggingface_hub','safetensors']},
    'repository_verification': verification['runtime'],
}
(RUN_DIR / 'runtime_environment.json').write_text(json.dumps(runtime, indent=2, sort_keys=True) + '\n')
print(json.dumps(runtime, indent=2))
assert torch.cuda.is_available(), 'A CUDA Colab runtime is required for the final training run.'


In [ ]:
# Cell 8 — Load the frozen tokenizer and train/validation files (no held-out content).
from datasets import load_dataset
from transformers import AutoTokenizer
training_cfg = json.loads((REPOSITORY_ROOT / 'experiments/qlora_patch_pilot_v1/training_config.json').read_text())
model_id = training_cfg['model_repository']
model_revision = training_cfg['model_revision']
tokenizer = AutoTokenizer.from_pretrained(model_id, revision=model_revision, cache_dir=str(MODEL_CACHE), use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
raw_train = load_dataset('json', data_files=str(TRAIN_JSONL), split='train')
raw_validation = load_dataset('json', data_files=str(VALIDATION_JSONL), split='train')

def tokenize_completion_only(example):
    prompt_text = tokenizer.apply_chat_template(example['prompt'], tokenize=False, add_generation_prompt=True)
    completion = example['completion']
    full_text = prompt_text + completion + (tokenizer.eos_token or '')
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)['input_ids']
    encoded = tokenizer(full_text, add_special_tokens=False, truncation=True, max_length=training_cfg['sft']['max_length'])
    labels = [-100] * min(len(prompt_ids), len(encoded['input_ids'])) + encoded['input_ids'][len(prompt_ids):]
    encoded['labels'] = labels
    return encoded

tokenized_train = raw_train.map(tokenize_completion_only, remove_columns=raw_train.column_names)
tokenized_validation = raw_validation.map(tokenize_completion_only, remove_columns=raw_validation.column_names)
assert len(tokenized_train) == 1000 and len(tokenized_validation) == 150
print({'train_examples': len(tokenized_train), 'validation_examples': len(tokenized_validation)})


In [ ]:
# Cell 9 — Load the pinned 7B checkpoint in frozen 4-bit QLoRA configuration.
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
quant = training_cfg['quantization']
quant_config = BitsAndBytesConfig(
    load_in_4bit=quant['load_in_4bit'],
    bnb_4bit_quant_type=quant['quant_type'],
    bnb_4bit_use_double_quant=quant['double_quant'],
    bnb_4bit_compute_dtype=compute_dtype,
)
model = AutoModelForCausalLM.from_pretrained(
    model_id, revision=model_revision, quantization_config=quant_config,
    device_map={'': 0}, cache_dir=str(MODEL_CACHE), dtype=compute_dtype,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
lora_cfg = training_cfg['lora']
lora = LoraConfig(
    r=lora_cfg['r'], lora_alpha=lora_cfg['alpha'], lora_dropout=lora_cfg['dropout'],
    target_modules=lora_cfg['target_modules'], bias=lora_cfg['bias'], task_type=lora_cfg['task_type'],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()


In [ ]:
# Cell 10 — Train exactly one epoch with the frozen training configuration (run-specific outputs).
from dataclasses import dataclass
from transformers import Trainer, TrainingArguments

@dataclass
class CompletionOnlyCollator:
    pad_token_id: int
    def __call__(self, features):
        max_len = max(len(item['input_ids']) for item in features)
        batch = {'input_ids': [], 'attention_mask': [], 'labels': []}
        for item in features:
            pad = max_len - len(item['input_ids'])
            batch['input_ids'].append(item['input_ids'] + [self.pad_token_id] * pad)
            batch['attention_mask'].append(item['attention_mask'] + [0] * pad)
            batch['labels'].append(item['labels'] + [-100] * pad)
        return {key: torch.tensor(value, dtype=torch.long) for key, value in batch.items()}

sft = training_cfg['sft']
final_adapter = RUN_DIR / 'adapter-final'
args = TrainingArguments(
    output_dir=str(RUN_DIR / 'trainer-output'),
    per_device_train_batch_size=sft['per_device_train_batch_size'],
    gradient_accumulation_steps=sft['gradient_accumulation_steps'],
    num_train_epochs=sft['num_train_epochs'],
    learning_rate=sft['learning_rate'],
    optim=sft['optim'],
    lr_scheduler_type=sft['lr_scheduler_type'],
    warmup_ratio=sft['warmup_ratio'],
    logging_steps=10,
    save_strategy='no',
    report_to='none',
    fp16=compute_dtype == torch.float16,
    bf16=compute_dtype == torch.bfloat16,
    seed=sft['seed'],
    data_seed=sft['data_seed'],
    gradient_checkpointing=sft['gradient_checkpointing'],
    gradient_checkpointing_kwargs={'use_reentrant': False},
)
trainer = Trainer(
    model=model, args=args,
    train_dataset=tokenized_train, eval_dataset=tokenized_validation,
    data_collator=CompletionOnlyCollator(tokenizer.pad_token_id),
)
train_started = time.time()
train_result = trainer.train()
elapsed_seconds = time.time() - train_started
peak_memory = {
    'peak_cuda_memory_allocated_bytes': torch.cuda.max_memory_allocated(),
    'peak_cuda_memory_reserved_bytes': torch.cuda.max_memory_reserved(),
}
print({'train_loss': train_result.training_loss, 'elapsed_seconds': elapsed_seconds, **peak_memory})


In [ ]:
# Cell 11 — Save adapter, tokenizer, trainer state, logs and the run provenance summary.
from agentic_debugger.training.patch_pilot import sha256_bytes
model.save_pretrained(final_adapter, safe_serialization=True)
tokenizer.save_pretrained(final_adapter)
trainer_state = {
    'global_step': trainer.state.global_step,
    'epoch': trainer.state.epoch,
    'log_history': trainer.state.log_history,
}
(RUN_DIR / 'trainer_state.json').write_text(json.dumps(trainer_state, indent=2, sort_keys=True) + '\n')
(RUN_DIR / 'training_log_history.json').write_text(json.dumps(trainer.state.log_history, indent=2, sort_keys=True) + '\n')
adapter_identities = {}
for path in sorted(final_adapter.rglob('*')):
    if path.is_file():
        adapter_identities[str(path.relative_to(final_adapter))] = {
            'size_bytes': path.stat().st_size,
            'sha256': sha256_bytes(path.read_bytes()),
        }
for name in ('trainer_state.json', 'training_log_history.json'):
    path = RUN_DIR / name
    print(name, path.stat().st_size, sha256_bytes(path.read_bytes()))
training_summary = {
    'schema_version': 'final-training-summary-v1',
    'experiment_id': 'qlora-patch-pilot-v1',
    'run_id': run_id,
    'run_status': 'PENDING_FINAL_VERIFICATION',
    'started_at_utc': run_context['created_at_utc'],
    'training_completed_at_utc': _dt.datetime.now(_dt.timezone.utc).isoformat(),
    'repository_commit': verification['runtime']['execution_head'],
    'required_ancestor': freeze['repository_baseline']['base_commit'],
    'authorization_path': str(AUTHORIZATION),
    'authorization_sha256': authorization_sha256,
    'completed_audit_csv_path': str(COMPLETED_AUDIT_CSV),
    'completed_audit_csv_sha256': auth_result['audit_artifact_identities']['completed_audit_csv']['sha256'],
    'completed_audit_manifest_path': str(COMPLETED_AUDIT_MANIFEST),
    'completed_audit_manifest_sha256': auth_result['audit_artifact_identities']['completed_audit_manifest']['sha256'],
    'train_jsonl_path': str(TRAIN_JSONL),
    'train_jsonl_sha256': auth_result['corpus_artifact_identities']['train_jsonl']['sha256'],
    'train_jsonl_bytes': auth_result['corpus_artifact_identities']['train_jsonl']['size_bytes'],
    'train_rows': auth_result['corpus_artifact_identities']['train_jsonl']['rows'],
    'validation_jsonl_path': str(VALIDATION_JSONL),
    'validation_jsonl_sha256': auth_result['corpus_artifact_identities']['validation_jsonl']['sha256'],
    'validation_jsonl_bytes': auth_result['corpus_artifact_identities']['validation_jsonl']['size_bytes'],
    'validation_rows': auth_result['corpus_artifact_identities']['validation_jsonl']['rows'],
    'corpus_manifest_path': str(CORPUS_MANIFEST),
    'corpus_manifest_sha256': auth_result['corpus_artifact_identities']['corpus_manifest']['sha256'],
    'model_repository': model_id,
    'model_revision': model_revision,
    'configuration_identities': auth_result['configuration_identities'],
    'audit_mode': auth_result['audit_mode'],
    'audit_result': auth_result['audit_counts'],
    'reviewer_identity': auth_result['reviewer_identity'],
    'reviewer_type': auth_result['reviewer_type'],
    'no_top_up': True,
    'train_loss': train_result.training_loss,
    'elapsed_seconds': elapsed_seconds,
    **peak_memory,
    'train_examples': 1000,
    'validation_examples': 150,
    'epochs': sft['num_train_epochs'],
    'trainer_state': trainer_state,
    'runtime': runtime,
    'adapter_identities': adapter_identities,
    'held_out_generation_authorized': False,
    'held_out_accessed': False,
}
(RUN_DIR / 'final_training_summary.json').write_text(json.dumps(training_summary, indent=2, sort_keys=True) + '\n')
print(json.dumps(training_summary, indent=2)[:3000])


In [ ]:
# Cell 12 — Reload the saved final adapter and verify it is usable.
import gc
from peft import PeftModel
del trainer, model
gc.collect(); torch.cuda.empty_cache()
base_model = AutoModelForCausalLM.from_pretrained(
    model_id, revision=model_revision, quantization_config=quant_config,
    device_map={'': 0}, cache_dir=str(MODEL_CACHE), dtype=compute_dtype,
)
reloaded_final = PeftModel.from_pretrained(base_model, final_adapter, is_trainable=False)
reloaded_final.eval()
reload_verification = {'adapter_reloaded': True, 'adapter_path': str(final_adapter), 'verified_at_utc': _dt.datetime.now(_dt.timezone.utc).isoformat()}
(RUN_DIR / 'reload_verification.json').write_text(json.dumps(reload_verification, indent=2, sort_keys=True) + '\n')
print(reload_verification)


In [ ]:
# Cell 13 — Final gates, run manifest, completion record. Status only after all steps.
from agentic_debugger.training.patch_pilot import write_external_manifest
assert reload_verification['adapter_reloaded'] is True, 'adapter reload verification failed'
assert auth_result['held_out_generation_authorized'] is False
assert freeze['scientific_gate']['held_out_generation_authorized'] is False
manifest = write_external_manifest(
    RUN_DIR,
    configuration_identity=freeze['training']['sha256'],
    provenance_identity=f'{model_id}@{model_revision}',
    artifact_kind_prefix='final-training',
)
completed_at_utc = _dt.datetime.now(_dt.timezone.utc).isoformat()
run_status = {
    'run_id': run_id,
    'status': 'COMPLETE',
    'final_status': 'FINAL_TRAINING_COMPLETE_AWAITING_FIRSTMATE_REVIEW',
    'completed_at_utc': completed_at_utc,
    'manifest_sha256': sha256_bytes((RUN_DIR / 'external_artifacts.json').read_bytes()),
}
(RUN_DIR / 'run_status.json').write_text(json.dumps(run_status, indent=2, sort_keys=True) + '\n')
(RUN_DIR / 'RUN_COMPLETE').write_text('COMPLETE run; see run_status.json\n', encoding='utf-8')
(RUN_DIR / 'INCOMPLETE').unlink()
training_summary['run_status'] = 'COMPLETE'
training_summary['final_status'] = run_status['final_status']
training_summary['completed_at_utc'] = completed_at_utc
training_summary['manifest_sha256'] = run_status['manifest_sha256']
training_summary['run_status_sha256'] = sha256_bytes((RUN_DIR / 'run_status.json').read_bytes())
training_summary['reload_verification'] = reload_verification
(RUN_DIR / 'final_training_summary.json').write_text(json.dumps(training_summary, indent=2, sort_keys=True) + '\n')
print(json.dumps(run_status, indent=2))
print('FINAL_TRAINING_COMPLETE_AWAITING_FIRSTMATE_REVIEW')


## Boundary — do not continue past this cell

No held-out task content was loaded or generated by this notebook. Held-out generation,
base-versus-tuned evaluation, dataset acquisition, and corpus modification remain forbidden
until FirstMate reviews the final adapter and `final_training_summary.json`. The run directory
is on Drive under `agentic-debugging/qlora_patch_pilot_v1/final-training/runs/<run-id>/`; a
completed adapter is accepted only after FirstMate reviews the complete provenance package.
